# Módulo 6: Segmentación de la banda de goma

Recibe la imagen rectificada (warped) del Módulo 4 y segmenta la banda de goma.

Estrategia:
1. Escala de grises + umbralización OTSU → zona oscura = goma.
2. Morfología (close + open) para rellenar huecos y eliminar ruido.
3. Selección del contorno más grande y suficientemente ancho → máscara final.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# Cargamos las funciones del notebook de QRs directamente
%run qr_detection.ipynb

IMAGE_PATH   = "../data/multimedia_01.jpg"
DATASET_PATH = "../data"
print("Imports OK")

## Función de segmentación

In [ ]:
def segment_rubber_band(warped_rgb, min_area_ratio=0.03, show=True):
    """
    Segmenta la banda de goma en una imagen ya rectificada.

    Returns
    -------
    mask     : ndarray H×W uint8, 255 donde está la goma
    contour  : contorno principal (np.ndarray Nx1x2) o None
    bbox     : (x, y, w, h) bounding rect del contorno, o None
    """
    H, W = warped_rgb.shape[:2]

    gray = cv2.cvtColor(warped_rgb, cv2.COLOR_RGB2GRAY)

    # OTSU sobre la imagen sin el borde negro del warp (que también es oscuro)
    # Recortamos un margen pequeño para evitar que el borde negro contamine OTSU
    margin = int(min(H, W) * 0.03)
    interior = gray[margin:H-margin, margin:W-margin]
    _, thresh_val = cv2.threshold(interior, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Aplicamos ese umbral a la imagen completa
    _, binary = cv2.threshold(gray, thresh_val if isinstance(thresh_val, (int, float))
                               else thresh_val, 255, cv2.THRESH_BINARY_INV)

    # Morfología: cerrar huecos internos y eliminar ruido pequeño
    k_close = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 25))
    k_open  = cv2.getStructuringElement(cv2.MORPH_RECT, (10, 10))
    binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close, iterations=2)
    binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open,  iterations=1)

    # Quitar el margen negro del borde del warp
    border_mask = np.zeros_like(binary)
    border_mask[margin:H-margin, margin:W-margin] = 255
    binary = cv2.bitwise_and(binary, border_mask)

    # Seleccionar el contorno más grande (la goma) si supera un área mínima
    cnts, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    min_area = H * W * min_area_ratio
    cnts = [c for c in cnts if cv2.contourArea(c) >= min_area]

    if not cnts:
        if show:
            plt.imshow(warped_rgb); plt.title("No se detectó goma"); plt.axis('off'); plt.show()
        return binary, None, None

    # Nos quedamos con el contorno más grande
    main_cnt = max(cnts, key=cv2.contourArea)
    mask = np.zeros((H, W), dtype=np.uint8)
    cv2.drawContours(mask, [main_cnt], -1, 255, -1)

    bbox = cv2.boundingRect(main_cnt)

    if show:
        _visualize_rubber(warped_rgb, mask, main_cnt, bbox)

    return mask, main_cnt, bbox


def _visualize_rubber(warped_rgb, mask, contour, bbox):
    overlay = warped_rgb.copy()
    colored = np.zeros_like(overlay)
    colored[mask == 255] = [255, 80, 0]          # naranja sobre la goma
    overlay = cv2.addWeighted(overlay, 0.6, colored, 0.4, 0)
    cv2.drawContours(overlay, [contour], -1, (255, 80, 0), 3)

    x, y, w, h = bbox
    cv2.rectangle(overlay, (x, y), (x+w, y+h), (0, 220, 255), 2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(warped_rgb); axes[0].set_title("Imagen rectificada"); axes[0].axis('off')
    axes[1].imshow(overlay);    axes[1].set_title(f"Segmentación goma  bbox=({x},{y},{w},{h})"); axes[1].axis('off')
    plt.tight_layout()
    plt.show()

print("Función segment_rubber_band lista.")

## Test sobre una imagen

In [ ]:
_, qrs = detect_qrs(IMAGE_PATH, show=False)
warped_rgb, H_norm, used = warp_workspace(IMAGE_PATH, qrs, output_size=(1000, 1400), show=True)

mask, contour, bbox = segment_rubber_band(warped_rgb, show=True)

if bbox:
    x, y, w, h = bbox
    area_px = int(cv2.contourArea(contour))
    print(f"Goma detectada: bbox=({x}, {y}, {w}, {h})  área={area_px} px²")
else:
    print("No se detectó la goma.")

## Evaluación sobre todo el dataset

In [ ]:
image_paths = sorted(glob.glob(os.path.join(DATASET_PATH, "*.jpg")))
print(f"Imágenes: {len(image_paths)}")

results = []
for path in image_paths:
    fname = os.path.basename(path)
    try:
        _, qrs_i = detect_qrs(path, show=False)
        if len(qrs_i) < 2:
            print(f"  [SKIP] {fname}: solo {len(qrs_i)} QRs")
            results.append((fname, False, None))
            continue
        warped_i, _, _ = warp_workspace(path, qrs_i, output_size=(1000, 1400), show=False)
        mask_i, cnt_i, bbox_i = segment_rubber_band(warped_i, show=False)
        ok = cnt_i is not None
        tag = 'OK  ' if ok else 'FAIL'
        area = int(cv2.contourArea(cnt_i)) if ok else 0
        print(f"  [{tag}] {fname}: área={area} px²  bbox={bbox_i}")
        results.append((fname, ok, bbox_i))
    except Exception as e:
        print(f"  [ERR ] {fname}: {e}")
        results.append((fname, False, None))

n_ok = sum(1 for _, ok, _ in results if ok)
print(f"\nSegmentación exitosa: {n_ok}/{len(image_paths)} ({100*n_ok/max(len(image_paths),1):.1f}%)")